# Pipeline — Chấm điểm an toàn phòng thí nghiệm

**Chuẩn bị:** upload cả thư mục dự án lên Google Drive, đổi tên thành `ChuyenDe` (không dấu), đường dẫn là `MyDrive/ChuyenDe`.
Bật GPU: *Runtime → Change runtime type → T4 GPU*.

| Bước | Khi nào chạy |
|---|---|
| 0. Cài đặt | Mỗi lần mở notebook |
| 1. Train bình chữa cháy | Hôm nay (5/10), khi đã có dataset Roboflow |
| 2–6 | Ngày mai, sau khi có ảnh |
| Demo | Bất cứ lúc nào, để kiểm tra pipeline |

In [ ]:
# 0. Cài đặt
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ChuyenDe
!pip install -q -r requirements.txt
!python src/test_rules.py

## 1. Train YOLO phát hiện bình chữa cháy
Trên Roboflow Universe: chọn dataset → *Download Dataset* → format **YOLOv8** → *show download code*. Dán đoạn code đó vào ô dưới, thay phần ví dụ.

Thời gian train khoảng 30–60 phút trên T4. Xong sẽ có `models/extinguisher_best.pt` và `outputs/train_metrics.json` (mAP để đưa vào bài).

In [ ]:
# --- DÁN ĐOẠN CODE TỪ ROBOFLOW VÀO ĐÂY (ví dụ) ---
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")
project = rf.workspace("WORKSPACE").project("PROJECT")
dataset = project.version(1).download("yolov8", location="/content/ext_dataset")
# -----------------------------------------------
!python src/train_extinguisher.py --data /content/ext_dataset/data.yaml --epochs 50

## 2. Vùng (lối đi, bàn, bình chữa cháy)
1. Copy ảnh vào `data/images/` (tên `v1_s01.jpg`, …).
2. Trên máy tính, mở `tools/zone_tool.html` bằng Chrome → mở ảnh → vẽ vùng cho mỗi góc → tải `zones.json` → copy vào `data/`.
3. Chạy ô dưới để kiểm tra vùng. Ảnh xem trước nằm trong `data/zones_preview/` (gửi kèm cho người chấm).

In [ ]:
!python src/preview_zones.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('data/zones_preview/*.jpg')):
    print(f); display(Image(f, width=600))

## 3. Tạo file chấm cho người chấm
Gửi `data/labels/raterX.csv` + `RUBRIC.md` + ảnh + `data/zones_preview/` cho từng người. Nhận lại file đã điền rồi chép đè vào `data/labels/`.

In [ ]:
!python src/make_label_sheets.py --raters 3

## 4. Cột 2 — Hệ thống luật (YOLO + bộ luật)
Kết quả: `outputs/results_rule.csv`, ảnh minh hoạ trong `outputs/annotated/`.

In [ ]:
!python src/run_rules.py
# Nếu YOLO-World lỗi khi cài CLIP: !python src/run_rules.py --no-world

## 5. Cột 3 — VLM (3 biến thể prompt)
Thêm API key vào **Colab Secrets** (biểu tượng chìa khoá bên trái): tên `GEMINI_API_KEY` (và/hoặc `ANTHROPIC_API_KEY`), bật *Notebook access*.
Chạy thử 2 ảnh trước, ổn rồi mới chạy toàn bộ. Bị ngắt giữa chừng thì cứ chạy lại, script sẽ tự bỏ qua phần đã xong.

In [ ]:
import os
from google.colab import userdata
for k in ['GEMINI_API_KEY', 'ANTHROPIC_API_KEY']:
    try: os.environ[k] = userdata.get(k)
    except Exception: pass

!python src/run_vlm.py --backend gemini --model gemini-2.5-flash --limit 2

In [ ]:
!python src/run_vlm.py --backend gemini --model gemini-2.5-flash
# Thêm Claude (tuỳ chọn, có phí):
# !python src/run_vlm.py --backend claude --model claude-haiku-4-5 --sleep 0.5

## 6. Đánh giá → bảng + hình cho bài báo
Kết quả trong `outputs/`: `table_*.csv/.tex`, `fig_kappa.png/.pdf`, `errors.csv`.

In [ ]:
!python src/evaluate.py
from IPython.display import Image, display
display(Image('outputs/fig_kappa.png', width=500))

## Demo — chạy thử bằng dữ liệu GIẢ (không dùng trong bài)

In [ ]:
!python src/make_demo_data.py
!python src/evaluate.py --labels data/demo/labels --rule data/demo/results_rule.csv --vlm data/demo/results_vlm.csv --out outputs/demo